# 🖼️ Glass Type Classification from Images using Deep Learning
 
## Project Overview
This notebook demonstrates how to classify glass types (e.g., container, tableware, headlamp, window) directly from images of broken glass using deep learning (CNN).
 
**Workflow:**
1. Import required libraries
2. Load and preprocess a sample image dataset (instructions for your own images included)
3. Build and train a Convolutional Neural Network (CNN)
4. Evaluate model performance
5. Interactive UI: Upload a glass image and get instant prediction
6. Guidance for expanding with your own dataset

## 1️⃣ Import Required Libraries
Import Python libraries for deep learning, image processing, and interactive UI.
- TensorFlow / Keras (for CNN)
- NumPy, Pandas (data handling)
- Matplotlib, Seaborn (visualization)
- PIL (image processing)
- ipywidgets (interactive UI)

In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML 

ModuleNotFoundError: No module named 'tensorflow'

## 2️⃣ Load and Explore the Image Dataset
You can use your own dataset of broken glass images, organized in folders by class (e.g., 'container', 'tableware', etc.).
 
For demonstration, we'll use a small sample dataset structure:
- `data/`
    - `container/`
    - `tableware/`
    - `headlamp/`
    - `window/`
 
Each folder should contain images of that glass type.

In [ ]:
# Load and Explore the Image Dataset
from tensorflow.keras.preprocessing import image_dataset_from_directory
 
# Set dataset path (update this to your actual path)
DATASET_PATH = 'data/'  # e.g., 'data/'
IMG_SIZE = (128, 128)
BATCH_SIZE = 16
 
# Check if dataset exists
if not os.path.exists(DATASET_PATH):
    print(f"❌ Dataset folder '{DATASET_PATH}' not found. Please add your images as described above.")
else:
    print(f"✅ Found dataset folder: {DATASET_PATH}")
    # Load dataset
    train_ds = image_dataset_from_directory(
        DATASET_PATH,
        labels='inferred',
        label_mode='categorical',
        batch_size=BATCH_SIZE,
        image_size=IMG_SIZE,
        shuffle=True,
        seed=42
    )
    # Show class names
    print(f"Classes found: {train_ds.class_names}")
    # Display a few sample images
    plt.figure(figsize=(10, 6))
    for images, labels in train_ds.take(1):
        for i in range(6):
            ax = plt.subplot(2, 3, i + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            plt.title(train_ds.class_names[np.argmax(labels[i])])
            plt.axis("off")
    plt.show()

## 3️⃣ Display Dataset Information
Show dataset info: number of classes, number of images per class, and image shape.

In [ ]:
# Display Dataset Information
if os.path.exists(DATASET_PATH):
    class_counts = {}
    for class_name in os.listdir(DATASET_PATH):
        class_dir = os.path.join(DATASET_PATH, class_name)
        if os.path.isdir(class_dir):
            count = len([f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
            class_counts[class_name] = count
    print(f"Number of classes: {len(class_counts)}")
    print(f"Images per class: {class_counts}")
    print(f"Image shape: {IMG_SIZE}")

## 4️⃣ Show Statistical Summary
Display a summary of the dataset: total images, class distribution, and example image shapes.

In [ ]:
# Show Statistical Summary
if os.path.exists(DATASET_PATH):
    total_images = sum(class_counts.values())
    print(f"Total images: {total_images}")
    print(f"Class distribution: {class_counts}")
    # Show example image shape
    for class_name in class_counts.keys():
        class_dir = os.path.join(DATASET_PATH, class_name)
        img_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if img_files:
            img_path = os.path.join(class_dir, img_files[0])
            img = Image.open(img_path)
            print(f"Example image from '{class_name}': shape = {img.size}, mode = {img.mode}")
            break

## 5️⃣ Visualize Distribution of Glass Types
Plot the count of each glass type using a bar chart.

In [ ]:
# Visualize Distribution of Glass Types
if os.path.exists(DATASET_PATH):
    plt.figure(figsize=(8, 5))
    plt.bar(class_counts.keys(), class_counts.values(), color='skyblue', edgecolor='black')
    plt.xlabel('Glass Type')
    plt.ylabel('Number of Images')
    plt.title('Distribution of Glass Types in Image Dataset')
    for i, v in enumerate(class_counts.values()):
        plt.text(i, v + 1, str(v), ha='center', fontweight='bold')
    plt.show()

## 6️⃣ Check for Missing Values
Check for missing or unreadable images in the dataset.

In [ ]:
# Check for Missing or Unreadable Images
if os.path.exists(DATASET_PATH):
    missing_images = 0
    for class_name in class_counts.keys():
        class_dir = os.path.join(DATASET_PATH, class_name)
        img_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        for img_file in img_files:
            img_path = os.path.join(class_dir, img_file)
            try:
                img = Image.open(img_path)
                img.verify()
            except Exception as e:
                print(f"Unreadable image: {img_path}")
                missing_images += 1
    print(f"Total unreadable images: {missing_images}")

## 7️⃣ Separate Features and Target
In image classification, features are image pixels and the target is the glass type label.

In [ ]:
# Features and Target for Image Classification
if os.path.exists(DATASET_PATH):
    print("Features: image pixels (shape: {} x {} x 3)".format(IMG_SIZE[0], IMG_SIZE[1]))
    print("Target: glass type label (one-hot encoded)")

## 8️⃣ Feature Correlation Analysis
For images, feature correlation is less interpretable, but you can visualize sample images per class.

In [ ]:
# Visualize Sample Images per Class
if os.path.exists(DATASET_PATH):
    plt.figure(figsize=(12, 6))
    for idx, class_name in enumerate(class_counts.keys()):
        class_dir = os.path.join(DATASET_PATH, class_name)
        img_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if img_files:
            img_path = os.path.join(class_dir, img_files[0])
            img = Image.open(img_path).resize(IMG_SIZE)
            ax = plt.subplot(1, len(class_counts), idx + 1)
            plt.imshow(img)
            plt.title(class_name)
            plt.axis('off')
    plt.suptitle('Sample Images per Glass Type')
    plt.show()

## 9️⃣ Feature Scaling
Image pixel values are typically scaled to [0, 1] for deep learning models.

In [ ]:
# Feature Scaling for Images
if os.path.exists(DATASET_PATH):
    normalization_layer = layers.Rescaling(1./255)
    print("Pixel values will be scaled to [0, 1] for model training.")

## 10️⃣ Split Data into Training and Testing Sets
Split the image dataset into training and validation sets for model evaluation.

In [ ]:
# Split Data into Training and Validation Sets
if os.path.exists(DATASET_PATH):
    train_ds = image_dataset_from_directory(
        DATASET_PATH,
        labels='inferred',
        label_mode='categorical',
        batch_size=BATCH_SIZE,
        image_size=IMG_SIZE,
        shuffle=True,
        seed=42,
        validation_split=0.2,
        subset='training'
    )
    val_ds = image_dataset_from_directory(
        DATASET_PATH,
        labels='inferred',
        label_mode='categorical',
        batch_size=BATCH_SIZE,
        image_size=IMG_SIZE,
        shuffle=True,
        seed=42,
        validation_split=0.2,
        subset='validation'
    )
    print(f"Training batches: {len(train_ds)}")
    print(f"Validation batches: {len(val_ds)}")

## 11️⃣ Build the CNN Model
Define a Convolutional Neural Network (CNN) for image classification.

In [ ]:
# Build the CNN Model
if os.path.exists(DATASET_PATH):
    num_classes = len(class_counts)
    model = models.Sequential([
        normalization_layer,
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=IMG_SIZE + (3,)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.summary()

## 12️⃣ Train the CNN Model
Train the CNN model on the training set and validate on the validation set.

In [ ]:
# Train the CNN Model
if os.path.exists(DATASET_PATH):
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        verbose=1
    )

## 13️⃣ Evaluate Model Performance
Plot training and validation accuracy and loss curves.

In [ ]:
# Plot Training and Validation Accuracy/Loss
if os.path.exists(DATASET_PATH):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    plt.show()

## 14️⃣ Interactive Glass Type Classifier
Upload an image of broken glass and get instant prediction using the trained CNN model.

In [ ]:
# Interactive Glass Type Classifier
def predict_image(img_bytes):
    img = Image.open(io.BytesIO(img_bytes)).resize(IMG_SIZE)
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    preds = model.predict(img_array)
    class_idx = np.argmax(preds[0])
    class_name = list(class_counts.keys())[class_idx]
    return class_name, preds[0][class_idx]
 
upload_widget = widgets.FileUpload(accept='image/*', multiple=False)
output_widget = widgets.Output()
 
def on_upload_change(change):
    with output_widget:
        clear_output()
        if upload_widget.value:
            uploaded_file = list(upload_widget.value.values())[0]
            img_bytes = uploaded_file['content']
            img = Image.open(io.BytesIO(img_bytes)).resize(IMG_SIZE)
            display(img)
            pred_class, confidence = predict_image(img_bytes)
            print(f"Predicted Glass Type: {pred_class} (Confidence: {confidence:.2f})")
 
upload_widget.observe(on_upload_change, names='value')
display(HTML('<h3>Upload a broken glass image for prediction:</h3>'))
display(upload_widget)
display(output_widget)